# Module 3: Prompt Engineering with LangChain

**Bano Qabil Generative AI Course**

**Prepared by Engr Usman Hasan**

This notebook continues directly from Module 2. In Module 2, you sent a question to an LLM and printed its answer. In Module 3, you will improve the instructions sent to the LLM.

Run the cells from top to bottom. Change one thing, run again, and compare the answer.

## Connection with Module 2

In Module 2, the main idea was:

1. Write a question.
2. Send the question to the model.
3. Read the answer.

Module 3 keeps the same idea. We now learn how to write better instructions and reuse them with LangChain.

## Block 1: Install LangChain

LangChain helps us organize messages and reuse prompts. Groq still provides the model.

In [ ]:
!pip install -qU langchain langchain-core langchain-groq python-dotenv


## Block 2: Load the API key in VS Code

Create a file named .env beside this notebook. Add this line:

GROQ_API_KEY=your_key_here

Never upload the .env file to GitHub.

In [ ]:
# ==========================================================
# BLOCK 2: LOAD THE API KEY FROM A LOCAL .env FILE
# Purpose: Keep the key outside the notebook
# ==========================================================
import os
from getpass import getpass
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("GROQ_API_KEY")

# Ask securely if the .env file is missing.
if not api_key:
    api_key = getpass("Enter your GROQ_API_KEY: ")

print("API key loaded safely.")


## Block 3: Create the LangChain model

This replaces the Groq client object from Module 2. The model call still uses your Groq API key.

In [ ]:
# ==========================================================
# BLOCK 3: CREATE THE LANGCHAIN CHAT MODEL
# Purpose: Connect LangChain to a Groq model
# ==========================================================
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

MODEL_NAME = "llama-3.3-70b-versatile"

# ChatGroq is LangChain's connection to the Groq model.
llm = ChatGroq(
    model=MODEL_NAME,
    api_key=api_key,
    temperature=0.2
)

print("LangChain model is ready:", MODEL_NAME)


## Block 4: Your first LangChain question

The invoke function sends the question. The answer text appears inside response.content.

In [ ]:
# ==========================================================
# BLOCK 4: ASK A SIMPLE QUESTION
# Purpose: Connect Module 2 knowledge with LangChain
# ==========================================================
user_prompt = "Explain solar energy to a 12-year-old student."

# Send the prompt through LangChain.
response = llm.invoke(user_prompt)

# Display only the generated text.
print(response.content)


## Topic 1: What is prompt engineering?

Prompt engineering means writing instructions that help the model understand:

- what job it should perform
- who will read the answer
- what information it should use
- what the answer should look like

A clear prompt usually produces a clearer answer.

In [ ]:
# Compare a weak prompt and a clear prompt.
weak_prompt = "Write about solar energy."

clear_prompt = """
Explain solar energy to a 14-year-old student.
Use simple English.
Write exactly 5 short sentences.
Give one everyday example.
"""

print("WEAK ANSWER:\n")
print(llm.invoke(weak_prompt).content)

print("\nCLEAR ANSWER:\n")
print(llm.invoke(clear_prompt).content)


## Topic 2: System and human messages

The system message gives the model its general role and rules.

The human message contains the current question.

The system message stays similar. The human message changes.

In [ ]:
# ==========================================================
# BLOCK 5: SYSTEM AND HUMAN MESSAGES
# Purpose: Separate general rules from the current question
# ==========================================================
messages = [
    (
        "system",
        "You are a patient teacher for beginner students. "
        "Use simple English and explain difficult words."
    ),
    (
        "human",
        "Explain voltage using a water pipe example."
    )
]

response = llm.invoke(messages)
print(response.content)


### System prompt experiment

Change only the system message:

- First use patient teacher.
- Then use funny science teacher.
- Then use strict examiner.

Keep the human question unchanged and compare the answers.

## Topic 3: Few-shot prompting

Few-shot prompting means showing the model a few examples before asking the new question.

The examples teach the pattern that we want.

In [ ]:
# ==========================================================
# BLOCK 6: FEW-SHOT PROMPTING
# Purpose: Teach a headline pattern by showing examples
# ==========================================================
few_shot_messages = [
    ("system", "Create one short and positive headline."),

    # Example 1
    ("human", "Topic: Saving water"),
    ("ai", "Every Drop Counts"),

    # Example 2
    ("human", "Topic: Learning Python"),
    ("ai", "Code Your First Idea"),

    # New question
    ("human", "Topic: Solar energy")
]

response = llm.invoke(few_shot_messages)
print(response.content)


### Few-shot challenge

Add one more example about road safety. Then ask for a headline about artificial intelligence.

Keep every example in the same format.

## Topic 4: Chain-of-thought prompting

For difficult tasks, the model can first make a short plan, then answer, then check the result.

In class we will use this simple visible format:

PLAN

ANSWER

CHECK

We only need a brief useful plan. We do not need hidden private reasoning.

In [ ]:
# ==========================================================
# BLOCK 7: PLAN, ANSWER, AND CHECK
# Purpose: Make a difficult task easier to review
# ==========================================================
reasoning_prompt = """
Create a short study plan for learning Python lists.

Return these three sections:
PLAN: Write 3 short steps.
ANSWER: Give the study plan.
CHECK: Confirm that the plan is suitable for a beginner.
"""

response = llm.invoke(reasoning_prompt)
print(response.content)


## Topic 5: JSON output

Normal text can appear in any shape. A program may struggle to find the title or hashtags.

JSON gives every value a name.

Example:

{
  "title": "Save Water",
  "caption": "Use water carefully every day.",
  "hashtags": ["#SaveWater", "#StudentAction"]
}

For now, remember only three ideas:

- Curly brackets hold the complete object.
- A key such as title names a value.
- Square brackets hold a list.

In [ ]:
# ==========================================================
# BLOCK 8: ASK FOR JSON USING A NORMAL PROMPT
# Purpose: See how a model follows a requested output format
# ==========================================================
json_request = """
Create a short social media post about saving electricity.

Return valid JSON only with these keys:
title
caption
hashtags

hashtags must be a list containing 2 items.
Do not add Markdown or extra explanation.
"""

response = llm.invoke(json_request)
print(response.content)


## JSON mode in LangChain

The prompt asks for JSON. JSON mode adds another instruction at the model setting level.

We still tell the model which keys we need.

In [ ]:
# ==========================================================
# BLOCK 9: TURN ON JSON MODE
# Purpose: Ask the model to return a valid JSON object
# ==========================================================
# bind creates a new version of the same LangChain model.
json_llm = llm.bind(
    response_format={"type": "json_object"}
)

# The words "JSON" and the required keys stay in the prompt.
json_prompt = """
Create a short post about recycling for students aged 14+.
Return one JSON object with these keys:
title, caption, hashtags
hashtags must be a list of 2 strings.
"""

json_response = json_llm.invoke(json_prompt)
print(json_response.content)


## Reading JSON in Python

The model returns JSON as text. Python can convert that text into a dictionary.

This step allows the program to access title, caption, and hashtags separately.

In [ ]:
# ==========================================================
# BLOCK 10: CONVERT JSON TEXT INTO A PYTHON DICTIONARY
# Purpose: Access each JSON value by its key
# ==========================================================
import json

# Convert the JSON text into a normal Python dictionary.
post_data = json.loads(json_response.content)

# Access each value using its key.
print("Title:", post_data["title"])
print("Caption:", post_data["caption"])
print("Hashtags:", " ".join(post_data["hashtags"]))


## Topic 6: Prompt templates

A prompt template is a reusable prompt with empty spaces called variables.

Examples of variables:

- topic
- audience
- tone
- platform

The instructions stay the same. Only the variable values change.

In [ ]:
# ==========================================================
# BLOCK 11: CREATE A REUSABLE PROMPT TEMPLATE
# Purpose: Reuse one prompt for different topics
# ==========================================================
content_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a content writer for beginner students. "
        "Use simple and safe language."
    ),
    (
        "human",
        "Create a {platform} post about {topic}. "
        "Audience: {audience}. Tone: {tone}. "
        "Return JSON with title, caption, and hashtags."
    )
])

# Join the prompt template and JSON model into one chain.
content_chain = content_prompt | json_llm


In [ ]:
# ==========================================================
# BLOCK 12: RUN THE SAME TEMPLATE WITH NEW VALUES
# Purpose: Generate content without rewriting the whole prompt
# ==========================================================
response = content_chain.invoke({
    "platform": "Instagram",
    "topic": "saving water",
    "audience": "students aged 14+",
    "tone": "friendly"
})

print(response.content)


In [ ]:
# Change only the values. The template stays unchanged.
second_response = content_chain.invoke({
    "platform": "LinkedIn",
    "topic": "learning Python",
    "audience": "beginner programmers",
    "tone": "encouraging"
})

print(second_response.content)


## Hands-on project: Simple Content Generator

Build a small application that asks the user for:

1. Topic
2. Audience
3. Platform
4. Tone

The application will use the prompt template, request JSON, convert it into a dictionary, and display the result.

In [ ]:
# ==========================================================
# STUDENT TASK: COMPLETE THE TODO LINES
# Do this before opening the teacher solution
# ==========================================================

# TODO 1: Ask the user for four inputs.
topic = input("Enter a topic: ")
audience = input("Enter the audience: ")
platform = input("Enter the platform: ")
tone = input("Enter the tone: ")

# TODO 2: Invoke content_chain with a dictionary.
# student_response = ...

# TODO 3: Convert student_response.content using json.loads().
# result = ...

# TODO 4: Print title, caption, and hashtags separately.


## Teacher solution

Try the student task first. Then compare your code with this solution.

In [ ]:
# ==========================================================
# COMPLETE SOLUTION: SIMPLE CONTENT GENERATOR
# ==========================================================
def generate_content(topic, audience, platform, tone):
    """Generate one JSON post and return it as a Python dictionary."""

    # Send the changing values into the reusable LangChain prompt.
    response = content_chain.invoke({
        "topic": topic,
        "audience": audience,
        "platform": platform,
        "tone": tone
    })

    # Convert the returned JSON text into a Python dictionary.
    return json.loads(response.content)


def display_content(data):
    """Display each part of the generated post clearly."""

    print("\nTITLE")
    print(data["title"])

    print("\nCAPTION")
    print(data["caption"])

    print("\nHASHTAGS")
    print(" ".join(data["hashtags"]))


In [ ]:
# Run the completed mini application.
topic = input("Enter a topic: ")
audience = input("Enter the audience: ")
platform = input("Enter the platform: ")
tone = input("Enter the tone: ")

try:
    result = generate_content(topic, audience, platform, tone)
    display_content(result)
except Exception as error:
    print("Something went wrong:", error)
    print("Check the API key, model name, internet, and JSON instructions.")


## Final challenges

1. Add a language variable to the prompt template.
2. Add a word limit variable.
3. Add one few-shot example before the final human message.
4. Generate posts for two different topics.
5. Change the output keys to title, description, and call_to_action.

Golden loop: change one thing, run again, read the output, and compare.

**Prepared by Engr Usman Hasan**